In [21]:
import pandas as pd

In [22]:
# Clamp-die lateral position should be deleted. since they just differ for 58 first experiment

In [23]:
import pandas as pd
import ast

# ============================================================
# Load CSV
# ============================================================
df = pd.read_csv("unique_bending_setups.csv")

# ============================================================
# Drop unused columns
# ============================================================
df.drop(columns=[
    'Outer-diameter',
    'Wall-thickness',
    'Target-angle',
    'Wiper-die shortening',
    'Mandrel position',
    'Tube_numbers',
    'Pressure-die lateral position',
    'Clamp-die lateral position'
], inplace=True)

# ============================================================
# Columns to exclude
# ============================================================
exclude_cols = ["Experiment_Number"]

# Columns to normalize
cols_to_normalize = df.columns.difference(exclude_cols)

# ============================================================
# Categorical Ordinal Normalization Function
# ============================================================
def categorical_normalize(series):
    """
    Convert discrete classes into ordinal normalized values in [0,1].

    Example: 4 unique values → 0, 0.33, 0.66, 1
    """
    classes = sorted(series.unique())
    n = len(classes)

    mapping = {
        cls: i / (n - 1)
        for i, cls in enumerate(classes)
    }

    return series.map(mapping), mapping


# ============================================================
# Apply Hybrid Normalization
# ============================================================
df = df.copy()

mappings = {}   # store categorical mappings

for col in cols_to_normalize:

    unique_vals = df[col].nunique()

    # ✅ Treat low-unique columns as categorical
    if unique_vals <= 10:

        df[col], mappings[col] = categorical_normalize(df[col])

    # ✅ Treat others as continuous
    else:
        df[col] = (
            (df[col] - df[col].min()) /
            (df[col].max() - df[col].min())
        )

# ============================================================
# Experiment_Number Handling
# ============================================================
df["Experiment_Number"] = df["Experiment_Number"].apply(ast.literal_eval)

# ============================================================
# Display Full DataFrame
# ============================================================
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None
):
    display(df)

# ============================================================
# Show categorical mappings (optional)
# ============================================================
print("\nCategorical mappings used:\n")
for col, mapping in mappings.items():
    print(f"{col}: {mapping}")


,Experiment_Number,Pressure-die distance,Pressure-die boost,Mandrel retraction timing,Collet boost
0,"[1, 2, 3]",0.0,0.0,0.333333,0.00
1,"[4, 5]",0.0,0.0,0.333333,0.00
2,"[6, 7]",0.0,0.0,0.333333,0.00
3,"[8, 9]",0.0,0.0,0.333333,0.00
4,"[10, 11]",0.0,0.0,0.333333,0.00
5,"[12, 13]",0.0,0.0,0.333333,0.00
6,"[14, 15, 50]",0.0,0.0,0.333333,0.00
7,"[16, 17]",0.0,0.0,0.333333,0.00
8,"[18, 19]",0.0,0.0,0.333333,0.00
9,"[20, 21, 51, 52, 53]",0.0,0.0,0.333333,0.00



Categorical mappings used:

Collet boost: {0.85: 0.0, 0.87: 0.25, 0.9: 0.5, 0.92: 0.75, 0.95: 1.0}
Mandrel retraction timing: {0.0: 0.0, 2.0: 0.3333333333333333, 5.0: 0.6666666666666666, 10.0: 1.0}
Pressure-die boost: {0.0: 0.0, 0.85: 0.2, 0.87: 0.4, 0.9: 0.6, 0.92: 0.8, 0.95: 1.0}
Pressure-die distance: {0.3: 0.0, 0.6: 0.5, 0.9: 1.0}


In [24]:
collet_boost = df['Collet boost'].unique()
mandrel_retraction = df['Mandrel retraction timing'].unique()
collet_boost, mandrel_retraction

(array([0.  , 0.25, 0.5 , 0.75, 1.  ]),
 array([0.33333333, 0.        , 0.66666667, 1.        ]))

In [25]:
df['Collet boost'].nunique(), df['Mandrel retraction timing'].nunique()

(5, 4)

In [26]:
# assume your dataframe is named df
df_expanded = (
    df
    .explode("Experiment_Number")
    .rename(columns={"Experiment_Number": "Experiment_ID"})
    .reset_index(drop=True)
)
df_expanded.to_csv('experiment_setups.csv', index=False)

In [28]:
import json
import ast
import pandas as pd

import numpy as np
import pandas as pd
import ast

def to_int_list(x):
    # handle list/array first
    if isinstance(x, (list, tuple, np.ndarray)):
        return [int(v) for v in x]

    # now safe to check scalar NA
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []

    if isinstance(x, str):
        parsed = ast.literal_eval(x)
        return [int(v) for v in parsed]

    if isinstance(x, (int, float)):
        return [int(x)]

    raise ValueError(f"Unexpected Experiment_Number value/type: {x!r} ({type(x)})")


df["Experiment_Number"] = df["Experiment_Number"].apply(to_int_list)

# -----------------------------
# 3) Your predefined groups (list of list[int])
# -----------------------------
groups = [
    [2, 3], [4, 5], [6, 7], [8, 9], [10, 11], [12, 13], [14, 15, 50],
    [16, 17], [18, 19], [20, 21, 51, 52, 53], [22, 23, 54], [55],
    [24, 25, 43, 45, 46, 47], [56], [26, 27], [28, 29], [30, 31, 44],
    [32, 33], [34, 35], [36, 37], [38], [39], [40], [41], [42], [57], [49],
    [193, 194, 195], [268, 269, 270], [241, 242, 243], [253, 257, 259],
    [196, 197, 198], [223, 224, 225], [265, 266, 267], [238, 239, 240],
    [254, 256, 260], [220, 221, 222], [208, 209, 210], [262, 263, 264],
    [235, 236, 237], [255, 258, 261], [205, 206, 207], [190, 191, 192],
    [280, 281, 282, 283, 284], [214, 215, 216], [211, 212, 213],
    [271, 272, 273], [274, 275, 276], [277, 278, 279], [290, 291, 292],
    [293, 294, 295], [296, 297, 298], [299, 300, 301], [226, 227, 228],
    [302, 303, 304, 317, 318], [229, 230, 231], [232, 233, 234],
    [305, 306, 307], [308, 309, 310], [311, 312, 313], [314, 315, 316],
    [244, 247, 250], [245, 248, 251], [246, 249, 252], [199, 200, 201],
    [285, 286, 287, 288, 289], [217, 218, 219], [202, 203, 204],
    [59, 90, 129, 152], [88, 115, 184, 185], [69, 103, 134, 154],
    [83, 110, 182, 183], [70, 92, 135, 165], [60, 99, 130, 151],
    [87, 114, 186, 187], [68, 102, 133, 155], [84, 111, 180, 181],
    [71, 93, 136, 164], [61, 100, 131, 150, 153], [86, 113, 188, 189],
    [67, 101, 132, 156], [85, 112, 178, 179], [72, 94, 137, 163],
    [58, 89, 146, 147], [119, 120, 121, 122, 123], [62, 98, 145, 148],
    [63, 97, 144, 149], [77, 116, 168, 174], [78, 117, 167, 173],
    [79, 118, 172], [64, 76, 104, 141, 159], [65, 105, 142, 158],
    [66, 106, 143, 157], [82, 109, 169, 175], [81, 108, 170, 176],
    [80, 107, 171, 177], [75, 91, 140, 160],
    [124, 125, 126, 127, 128], [74, 96, 139, 161], [73, 95, 138, 162],
]

# -----------------------------
# 4) Build test_set based on your conditions
#    IMPORTANT: use & and reference the df columns on both sides
# -----------------------------
mask = (df["Collet boost"] == 0.25)

# Flatten lists-of-ints from the selected rows into a set[int]
test_set = {n for lst in df.loc[mask, "Experiment_Number"] for n in lst}

# -----------------------------
# 5) Split groups
# -----------------------------
train_groups = []
test_groups = []

for group in groups:
    if any(num in test_set for num in group):
        test_groups.append(group)
    else:
        train_groups.append(group)

# -----------------------------
# 6) Save result as JSON
# -----------------------------
result = {"train_groups": train_groups, "test_groups": test_groups}

print("Selected experiments in test_set:", len(test_set))
print("train_groups:", len(train_groups), "test_groups:", len(test_groups))

with open("train_test_split.json", "w") as f:
    json.dump(result, f, indent=2)


Selected experiments in test_set: 42
train_groups: 88 test_groups: 12
